### Setup

Import required libraries for STAC access, date handling, geopandas, and raster statistics, and define functions for querying STAC items and extracting pixel values from queried STAC items.

In [1]:
# import libraries

import os
import time
from datetime import datetime, timezone
from requests.exceptions import RequestException

import geopandas as gpd
from rasterstats import point_query, zonal_stats
from pystac_client import Client


Define the STACQueryClient helper class for querying collections, filtering items, and handling retries.

In [3]:
STAC_URL = "https://maps.landscape-geoinformatics.org/stac"

In [4]:
# define STAC query client function
class STACQueryClient:
    def __init__(self, stac_url=STAC_URL, max_retries=3, backoff=2):
        self.stac_url = stac_url
        self.max_retries = max_retries
        self.backoff = backoff
        self.catalog = Client.open(self.stac_url)

        # Cache all available collections at initialization
        self._collections_cache = set(
            [c.id for c in self.catalog.get_collections()]
        )

    # Collection resolution
    def build_collection_candidates(self, region, source, variable=None):
        """
        Try variable as catalog suffix first, then fallback to base collection.
        """
        candidates = []

        if variable:
            candidates.append(f"{region}-{source}-{variable}")

        candidates.append(f"{region}-{source}")

        return candidates

    def collection_exists(self, collection_id):
        return collection_id in self._collections_cache

    def resolve_collection(self, region, source, variable=None):
        """
        Return the first valid collection from candidates.
        """
        candidates = self.build_collection_candidates(region, source, variable)

        for c in candidates:
            if self.collection_exists(c):
                return c

        raise ValueError(
            f"No valid collection found for region='{region}', source='{source}', variable='{variable}'"
        )

    # Helpers
    def parse_dt(self, dt):
        """
        Convert any datetime string into a timezone-aware UTC datetime.
        """
        if isinstance(dt, datetime):
            if dt.tzinfo is None:
                return dt.replace(tzinfo=timezone.utc)
            return dt.astimezone(timezone.utc)

        dt_obj = datetime.fromisoformat(dt)

        if dt_obj.tzinfo is None:
            return dt_obj.replace(tzinfo=timezone.utc)

        return dt_obj.astimezone(timezone.utc)
    
    def interval_overlap(self, item_start, item_end, q_start, q_end):
        return item_end >= q_start and item_start <= q_end

    # Keyword matching
    
    def matches_keywords(self, item_id, variable=None, stat=None):
        item_id = item_id.lower()

        # handle variable (string or list)
        if variable:
            if isinstance(variable, str):
                variable_list = [variable.lower()]
            else:
                variable_list = [v.lower() for v in variable]

            if not any(v in item_id for v in variable_list):
                return False

        # handle stat (string or list)
        if stat:
            if isinstance(stat, str):
                stat_list = [stat.lower()]
            else:
                stat_list = [s.lower() for s in stat]

            if not any(f"_{s}_" in item_id for s in stat_list):
                return False

        return True
        
    # Time filtering (interval-based)
    def matches_time(self, item, date_range):
        if not date_range:
            return True

        props = item.properties

        start = props.get("start_datetime")
        end = props.get("end_datetime")

        if not start or not end:
            return False

        item_start = self.parse_dt(start)
        item_end = self.parse_dt(end)

        q_start_str, q_end_str = date_range.split("/")
        q_start = self.parse_dt(q_start_str)
        q_end = self.parse_dt(q_end_str)

        return self.interval_overlap(item_start, item_end, q_start, q_end)

    # Main search
    def search(
        self,
        region,
        source,
        variable=None,
        stat=None,
        date_range=None,
        max_items=None
    ):
        """
        Unified search:
        - variable is used first as catalog selector
        - fallback to keyword filtering if needed
        """

        collection_id = self.resolve_collection(region, source, variable)

        attempt = 0
        while attempt < self.max_retries:
            try:
                search = self.catalog.search(
                    collections=[collection_id],
                    datetime=date_range,
                    max_items=max_items
                )

                results = []

                for item in search.items():

                    # Keyword + stat filtering
                    if not self.matches_keywords(item.id, variable, stat):
                        continue

                    # Time filtering
                    if not self.matches_time(item, date_range):
                        continue

                    results.append(item)

                return results

            except RequestException as e:
                attempt += 1
                wait = self.backoff ** attempt
                print(f"Retry {attempt}/{self.max_retries} after error: {e}")
                time.sleep(wait)

        raise RuntimeError("STAC query failed after retries")

In [6]:
# define the point value extraction function
def extract_point_values(
    client,
    points_gdf,
    region,
    source,
    variables,
    stat=None,
    date_range=None
):
    """
    Extract raster values for point geometries and attach them to a GeoDataFrame.

    Returns an enriched GeoDataFrame (no export, no reshaping).
    """

    

    if isinstance(variables, str):
        variables = [variables]

    if isinstance(stat, str):
        stat = [stat]

    gdf = points_gdf.copy().reset_index(drop=True)

    # Unique point ID to preserve alignment
    gdf["point_id"] = gdf.index

    for var in variables:

        # Query items once per variable
        items = client.search(
            region=region,
            source=source,
            variable=var,
            stat=stat,
            date_range=date_range
        )

        for item in items:

            # Asset resolution
            asset = item.assets.get(var)

            if not asset:
                if item.assets:
                    asset = list(item.assets.values())[0]
                else:
                    print(f"Skipping {item.id}: no assets found")
                    continue

            raster_url = asset.href

            # Extract raster values at point locations
            values = point_query(
                vectors=gdf,
                raster=raster_url,
                interpolate="nearest"
            )

            # Store as new column in GDF
            
            col_name = f"{os.path.splitext(item.id)[0]}"

            gdf[col_name] = values

    return gdf


# define the zonal stats extraction function
def extract_zonal_stats(
    client,
    polygons_gdf,
    region,
    source,
    variables,
    stat=None,              # temporal stat (STAC)
    zonal_stat=None,        # spatial stat (rasterstats)
    date_range=None
):
    """
    Extract zonal statistics from STAC rasters into a GeoDataFrame.

    stat        -> temporal aggregation (STAC filtering)
    zonal_stat  -> spatial aggregation (zonal_stats)
    """

    if isinstance(variables, str):
        variables = [variables]

    if isinstance(zonal_stat, str):
        zonal_stat = [zonal_stat]

    gdf = polygons_gdf.copy().reset_index(drop=True)
    gdf["poly_id"] = gdf.index

    for var in variables:

        # STAC query (temporal stat)
        items = client.search(
            region=region,
            source=source,
            variable=var,
            stat=stat,
            date_range=date_range
        )

        for item in items:

            # asset resolution
            asset = item.assets.get(var)

            if not asset:
                if item.assets:
                    asset = list(item.assets.values())[0]
                else:
                    print(f"Skipping {item.id}: no assets found")
                    continue

            raster_url = asset.href

            # zonal stats (spatial stat)
            zs = zonal_stats(
                vectors=gdf.geometry,
                raster=raster_url,
                stats=zonal_stat if zonal_stat else ["mean"]
            )

            # attach results
            for stat_name in (zonal_stat if zonal_stat else ["mean"]):
                values = [z.get(stat_name) for z in zs]

                import os
                zs_name = zonal_stat if isinstance(zonal_stat, str) else "_".join(zonal_stat)
                item_id_clean = os.path.splitext(item.id)[0]
                col_name = f"{item_id_clean}_{zs_name}"
                #col_name = f"{var}_{stat}_{stat_name}_{os.path.splitext(item.id)[0]}"
                gdf[col_name] = values

    return gdf

#### Initialize STAC client
Set the STAC catalog endpoint and create a query client instance for search operations.

In [ ]:
# this might take a moment on the furst run due to collection caching

client = STACQueryClient()

c:\Users\fauzan\micromamba\envs\geopy\lib\site-packages\pystac_client\client.py:787: MissingLink: No link with rel='data' could be found on this Client.
  href = self._get_href("data", data_link, "collections")


### Example 1: Querying STAC item

#### Sentinel-2
Search the Sentinel-2 NDVI items with median and mean aggregation over the selected date range.

In [14]:
items = client.search(
    region="estonia",
    source="sentinel2",
    variable="ndvi",
    stat=["median", "mean"],               # indicates temporal composite
    date_range="2018-04-01/2018-08-31"
)

for item in items:
    print(item.id)

est_s2_ndvi_mean_2018-04-01_2018-05-31_cog.tif
est_s2_ndvi_mean_2018-06-01_2018-08-31_cog.tif
est_s2_ndvi_median_2018-04-01_2018-05-31_cog.tif
est_s2_ndvi_median_2018-06-01_2018-08-31_cog.tif


#### DEM derivatives
Search the topographic collections for hillshade and DEM assets.

In [9]:
items = client.search(
    region="estonia",
    source="topo",
    variable=["hillshade", "dem"]
)

for item in items:
    print(item.id)

dem_10m_clipped_cog.tif
hillshade_10m_clipped_cog.tif


#### Temperature
Search ERA5 mean temperature items for the defined region.

In [12]:
items = client.search(
    region="estonia",
    source="era5",
    variable="meantemp",
)

for item in items:
    print(item.id)

est_era5_meantemp_2017.tif
est_era5_meantemp_2018.tif
est_era5_meantemp_2019.tif
est_era5_meantemp_2020.tif
est_era5_meantemp_2021.tif
est_era5_meantemp_2022.tif
est_era5_meantemp_2023.tif
est_era5_meantemp_2024.tif


### Example 2: Extract pixel values at points

#### Load point geometries
Read the point geospatial dataset for pixel value extraction.

In [15]:
points_gdf = gpd.read_file("D:/demo_point.gpkg")

#### Extract pixel values of Satellite image derivatives
Use the point locations to extract NDVI and NDMI values from Sentinel-2 assets.

In [16]:
gdf_out = extract_point_values(
    client=client,
    points_gdf=points_gdf,
    region="estonia",
    source="sentinel2",
    variables=["ndvi", "ndmi"],
    stat=["median", "mean"],     # indicates temporal composite
    date_range="2018-04-01/2018-05-31"
)

gdf_out.head()

,id,type,geometry,point_id,est_s2_ndvi_mean_2018-04-01_2018-05-31_cog,est_s2_ndvi_median_2018-04-01_2018-05-31_cog,est_s2_ndmi_mean_2018-04-01_2018-05-31_cog,est_s2_ndmi_median_2018-04-01_2018-05-31_cog
0,1,field island,POINT (632084.69 6553633.435),0,426,403,-2,-83
1,2,arable land,POINT (632246.002 6553289.77),1,244,253,-64,-149
2,3,water,POINT (629597.196 6549949.798),2,-34,2,76,-25
3,4,forest,POINT (631217.084 6554656.593),3,618,610,215,299
4,5,built-up,POINT (628624.995 6554793.371),4,150,144,-71,-126


#### Extract pixel values of DEM derivatives
Use the same points to extract DEM and hillshade values from the topographic collection.

In [17]:
gdf_out = extract_point_values(
    client=client,
    points_gdf=points_gdf,
    region="estonia",
    source="topo",
    variables=["dem", "hillshade"]
)

gdf_out.head()

,id,type,geometry,point_id,dem_10m_clipped_cog,hillshade_10m_clipped_cog
0,1,field island,POINT (632084.69 6553633.435),0,109.939026,195
1,2,arable land,POINT (632246.002 6553289.77),1,108.679024,196
2,3,water,POINT (629597.196 6549949.798),2,98.083054,195
3,4,forest,POINT (631217.084 6554656.593),3,109.609962,197
4,5,built-up,POINT (628624.995 6554793.371),4,108.919693,195


#### Export point results
Save the point extraction results to GeoPackage and CSV formats.

In [ ]:
# save points with extracted values as geopackage or csv

gdf_out.to_file("points_output.gpkg", driver="GPKG")
gdf_out.to_csv("points_output.csv", index=False)

### Example 3: Extract zonal statistics at areas

#### Load polygon geometries
Read the polygon dataset used for zonal statistics extraction.

In [18]:
polygons_gdf = gpd.read_file("D:/demo_area.gpkg")

#### Extract zonal statistics for satellite derivatives
Calculate zonal statistics for NDVI and NDMI from Sentinel-2 using median spatial aggregation.

In [19]:
gdf_out = extract_zonal_stats(
    client=client,
    polygons_gdf=polygons_gdf,
    region="estonia",
    source="sentinel2",
    variables=["ndvi", "ndmi"],
    stat=["median", "mean"],     # indicates temporal composite
    zonal_stat="median",         # indicates spatial aggregation method for zonal stats
    date_range="2018-04-01/2018-05-31"
)

gdf_out.head()

,id,type,geometry,poly_id,est_s2_ndvi_mean_2018-04-01_2018-05-31_cog_median,est_s2_ndvi_median_2018-04-01_2018-05-31_cog_median,est_s2_ndmi_mean_2018-04-01_2018-05-31_cog_median,est_s2_ndmi_median_2018-04-01_2018-05-31_cog_median
0,1,field island,"POLYGON ((632084.69 6553652.051, 632085.015 65...",0,402.0,381.0,-4.0,-83.5
1,2,arable land,"POLYGON ((632246.002 6553308.387, 632246.327 6...",1,244.0,253.0,-74.0,-156.0
2,3,water,"POLYGON ((629597.196 6549968.415, 629597.521 6...",2,-36.0,-13.5,76.0,-26.5
3,4,forest,"POLYGON ((631217.084 6554675.209, 631217.409 6...",3,613.0,599.0,205.0,294.0
4,5,built-up,"POLYGON ((628624.995 6554811.988, 628625.32 65...",4,149.0,141.0,-54.0,-110.0


#### Extract zonal statistics for DEM derivatives
Calculate zonal statistics for DEM and hillshade assets from the topographic collection.

In [20]:
gdf_out = extract_zonal_stats(
    client=client,
    polygons_gdf=polygons_gdf,
    region="estonia",
    source="topo",
    variables=["dem", "hillshade"],
    zonal_stat="median"     # indicates spatial aggregation method for zonal stats
)

gdf_out.head()

,id,type,geometry,poly_id,dem_10m_clipped_cog_median,hillshade_10m_clipped_cog_median
0,1,field island,"POLYGON ((632084.69 6553652.051, 632085.015 65...",0,109.348541,194.0
1,2,arable land,"POLYGON ((632246.002 6553308.387, 632246.327 6...",1,108.679024,196.0
2,3,water,"POLYGON ((629597.196 6549968.415, 629597.521 6...",2,98.085678,195.0
3,4,forest,"POLYGON ((631217.084 6554675.209, 631217.409 6...",3,109.670631,197.0
4,5,built-up,"POLYGON ((628624.995 6554811.988, 628625.32 65...",4,108.886444,195.5


#### Export zonal statistics results
Save the polygon-level zonal statistics to GeoPackage and CSV.

In [ ]:
# save polygons with zonal stats as geopackage or csv

gdf_out.to_file("polygons_output.gpkg", driver="GPKG")
gdf_out.to_csv("polygons_output.csv", index=False)